<a href="https://colab.research.google.com/github/nurhikmahsalam3-creator/Tugas-4-Sistem-temu-kembali/blob/main/240210500015_NUR_HIKMAH_SALAM_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NO 2 - PEMBOBOTAN (TERM WEIGHTING)**

TOPIK: Bag-of-Words, TF, IDF, dan TF-IDF

In [2]:
### Instalasi & Import Library

# Install Sastrawi untuk stopword remover Bahasa Indonesia (dipakai di tahap preprocessing sederhana)
!pip install Sastrawi -q

import re     # untuk membersihkan teks (menghapus tanda baca/angka) dengan regular expression
import math   # untuk fungsi logaritma, dipakai pada rumus IDF
import pandas as pd  # untuk menampilkan data dalam bentuk tabel (DataFrame) yang rapi
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer  # tool siap pakai untuk menghitung TF-IDF otomatis

# Ambil daftar stopwords Bahasa Indonesia bawaan Sastrawi
stopword_factory = StopWordRemoverFactory()
stopwords_set = set(stopword_factory.get_stop_words())

In [3]:
### DATASET

# 4 dokumen pendek (1-3 kalimat) yang akan direpresentasikan menjadi vektor TF-IDF
dokumen = [
    "Sistem komputer modern menggunakan prosesor yang sangat cepat untuk mengolah data.",
    "Jaringan komputer menghubungkan banyak perangkat sehingga dapat saling bertukar data.",
    "Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen relevan dari data yang besar."
]

for i, d in enumerate(dokumen, 1):
    print(f"Dokumen {i}: {d}")

Dokumen 1: Sistem komputer modern menggunakan prosesor yang sangat cepat untuk mengolah data.
Dokumen 2: Jaringan komputer menghubungkan banyak perangkat sehingga dapat saling bertukar data.
Dokumen 3: Kecerdasan buatan adalah cabang ilmu komputer yang mempelajari cara membuat mesin cerdas.
Dokumen 4: Sistem temu kembali informasi membantu pengguna menemukan dokumen relevan dari data yang besar.


In [4]:
## Preprocessing Sederhana (Case Folding + Tokenisasi + Stopwords Removal)

def simple_preprocess(text):
    """Preprocessing minimal: case folding -> cleaning -> tokenisasi -> stopwords removal."""
    text = text.lower()                          # case folding: samakan huruf jadi huruf kecil
    text = re.sub(r"[^a-z\s]", " ", text)         # hapus karakter selain huruf & spasi (tanda baca, angka)
    text = re.sub(r"\s+", " ", text).strip()      # rapikan spasi berlebih
    tokens = text.split()                         # tokenisasi: pecah jadi daftar kata
    tokens = [t for t in tokens if t not in stopwords_set]  # buang kata-kata umum (stopwords)
    return tokens

# Terapkan preprocessing ke seluruh dokumen
tokenized_docs = [simple_preprocess(d) for d in dokumen]  # hasil: list token per dokumen
clean_docs = [" ".join(tokens) for tokens in tokenized_docs]  # gabungkan lagi jadi string, untuk input TfidfVectorizer

for i, (tok, clean) in enumerate(zip(tokenized_docs, clean_docs), 1):
    print(f"Dok {i} token: {tok}")


Dok 1 token: ['sistem', 'komputer', 'modern', 'menggunakan', 'prosesor', 'sangat', 'cepat', 'mengolah', 'data']
Dok 2 token: ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'saling', 'bertukar', 'data']
Dok 3 token: ['kecerdasan', 'buatan', 'cabang', 'ilmu', 'komputer', 'mempelajari', 'cara', 'membuat', 'mesin', 'cerdas']
Dok 4 token: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan', 'data', 'besar']


In [5]:
### Representasi Bag-of-Words (Raw Count)

# Bangun vocabulary = seluruh term unik dari semua dokumen, diurutkan alfabetis
vocab = sorted(set(t for tokens in tokenized_docs for t in tokens))

# Untuk setiap dokumen, hitung berapa kali tiap term di vocabulary muncul (raw count)
bow_rows = []
for i, tokens in enumerate(tokenized_docs, 1):
    row = {"Dokumen": f"Dok {i}"}
    for term in vocab:
        row[term] = tokens.count(term)   # hitung kemunculan term dalam dokumen ke-i
    bow_rows.append(row)

df_bow = pd.DataFrame(bow_rows).set_index("Dokumen")  # ubah jadi tabel, kolom "Dokumen" jadi index
df_bow


,banyak,bertukar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,...,mesin,modern,pengguna,perangkat,prosesor,relevan,saling,sangat,sistem,temu
Dokumen,,,,,,,,,,,,,,,,,,,,,
Dok 1,0,0,0,0,0,0,1,0,1,0,...,0,1,0,0,1,0,0,1,1,0
Dok 2,1,1,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,1,0,0,0
Dok 3,0,0,0,1,1,1,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
Dok 4,0,0,1,0,0,0,0,0,1,1,...,0,0,1,0,0,1,0,0,1,1


In [6]:
### Perhitungan Manual: TF, DF, dan IDF

# --- Term Frequency (TF) = raw count, nilainya sama seperti tabel BoW di atas ---
df_tf_manual = df_bow.copy()
print("Term Frequency (TF) per dokumen:")
df_tf_manual

# --- Document Frequency (df) dan Inverse Document Frequency (IDF) ---
N = len(dokumen)  # jumlah total dokumen dalam koleksi

# df(t) = jumlah DOKUMEN yang mengandung term t (bukan jumlah kemunculan totalnya)
df_counts = {}
for term in vocab:
    df_counts[term] = sum(1 for tokens in tokenized_docs if term in tokens)

# IDF versi smooth (mirip default scikit-learn): idf = ln((1+N)/(1+df)) + 1
# tujuan "+1" (smoothing) adalah mencegah pembagian dengan nol jika ada term yang df-nya 0
idf_values = {}
for term in vocab:
    df_t = df_counts[term]
    idf_values[term] = math.log((1 + N) / (1 + df_t)) + 1

# Susun df(t) dan IDF ke dalam tabel agar mudah dibaca
df_idf = pd.DataFrame({
    "term": vocab,
    "df(t)": [df_counts[t] for t in vocab],
    "IDF (smooth)": [round(idf_values[t], 4) for t in vocab]
}).set_index("term")

df_idf


Term Frequency (TF) per dokumen:


,df(t),IDF (smooth)
term,,
banyak,1,1.9163
bertukar,1,1.9163
besar,1,1.9163
buatan,1,1.9163
cabang,1,1.9163
cara,1,1.9163
cepat,1,1.9163
cerdas,1,1.9163
data,3,1.2231


In [7]:
### Perhitungan Manual: Matriks TF-IDF

# TF-IDF(t, d) = TF(t, d) x IDF(t)
tfidf_manual_rows = []
for i, tokens in enumerate(tokenized_docs, 1):
    row = {"Dokumen": f"Dok {i}"}
    for term in vocab:
        tf = tokens.count(term)                       # TF term ini di dokumen ke-i
        row[term] = round(tf * idf_values[term], 4)   # kalikan dengan IDF term tersebut
    tfidf_manual_rows.append(row)

df_tfidf_manual = pd.DataFrame(tfidf_manual_rows).set_index("Dokumen")
print("Matriks TF-IDF (perhitungan MANUAL):")
df_tfidf_manual


Matriks TF-IDF (perhitungan MANUAL):


,banyak,bertukar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,...,mesin,modern,pengguna,perangkat,prosesor,relevan,saling,sangat,sistem,temu
Dokumen,,,,,,,,,,,,,,,,,,,,,
Dok 1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.9163,0.0000,1.2231,0.0000,...,0.0000,1.9163,0.0000,0.0000,1.9163,0.0000,0.0000,1.9163,1.5108,0.0000
Dok 2,1.9163,1.9163,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.2231,0.0000,...,0.0000,0.0000,0.0000,1.9163,0.0000,0.0000,1.9163,0.0000,0.0000,0.0000
Dok 3,0.0000,0.0000,0.0000,1.9163,1.9163,1.9163,0.0000,1.9163,0.0000,0.0000,...,1.9163,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Dok 4,0.0000,0.0000,1.9163,0.0000,0.0000,0.0000,0.0000,0.0000,1.2231,1.9163,...,0.0000,0.0000,1.9163,0.0000,0.0000,1.9163,0.0000,0.0000,1.5108,1.9163


In [8]:
### Implementasi dengan Scikit-learn (`TfidfVectorizer`)

# TfidfVectorizer otomatis melakukan tokenisasi, hitung TF, IDF, TF-IDF, dan normalisasi L2 sekaligus
vectorizer = TfidfVectorizer()

# fit_transform() = "fit" (pelajari vocabulary & IDF dari corpus) + "transform" (hitung TF-IDF tiap dokumen)
tfidf_matrix = vectorizer.fit_transform(clean_docs)

# Ubah hasil (sparse matrix) menjadi DataFrame agar mudah dibaca dan dibandingkan
df_tfidf_sklearn = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),   # nama kolom = term-term hasil vocabulary otomatis
    index=[f"Dok {i}" for i in range(1, N + 1)]
).round(4)

print("Matriks TF-IDF (SCIKIT-LEARN):")
df_tfidf_sklearn


Matriks TF-IDF (SCIKIT-LEARN):


,banyak,bertukar,besar,buatan,cabang,cara,cepat,cerdas,data,dokumen,...,mesin,modern,pengguna,perangkat,prosesor,relevan,saling,sangat,sistem,temu
Dok 1,0.0000,0.0000,0.0000,0.000,0.000,0.000,0.3667,0.000,0.2341,0.0000,...,0.000,0.3667,0.0000,0.0000,0.3667,0.0000,0.0000,0.3667,0.2891,0.0000
Dok 2,0.3831,0.3831,0.0000,0.000,0.000,0.000,0.0000,0.000,0.2445,0.0000,...,0.000,0.0000,0.0000,0.3831,0.0000,0.0000,0.3831,0.0000,0.0000,0.0000
Dok 3,0.0000,0.0000,0.0000,0.326,0.326,0.326,0.0000,0.326,0.0000,0.0000,...,0.326,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Dok 4,0.0000,0.0000,0.3328,0.000,0.000,0.000,0.0000,0.000,0.2124,0.3328,...,0.000,0.0000,0.3328,0.0000,0.0000,0.3328,0.0000,0.0000,0.2624,0.3328


In [9]:
### Perbandingan Hasil Manual vs Scikit-learn

# idxmax() mencari nama kolom (term) dengan nilai tertinggi pada tiap baris (dokumen)
print("=== Term dengan bobot TF-IDF tertinggi per dokumen ===\n")
for i in range(1, N + 1):
    top_manual = df_tfidf_manual.loc[f"Dok {i}"].idxmax()
    top_sklearn = df_tfidf_sklearn.loc[f"Dok {i}"].idxmax()
    print(f"Dok {i}: manual -> '{top_manual}' | scikit-learn -> '{top_sklearn}'")


=== Term dengan bobot TF-IDF tertinggi per dokumen ===

Dok 1: manual -> 'cepat' | scikit-learn -> 'cepat'
Dok 2: manual -> 'banyak' | scikit-learn -> 'banyak'
Dok 3: manual -> 'buatan' | scikit-learn -> 'buatan'
Dok 4: manual -> 'besar' | scikit-learn -> 'besar'


**ANALISIS**

Pada setiap dokumen, term dengan bobot TF-IDF tertinggi cenderung adalah term yang sering muncul di dokumen tersebut namun jarang muncul di
dokumen lain dalam koleksi, misalnya "prosesor" pada Dok 1, "jaringan" pada Dok 2, "kecerdasan" atau "cerdas" pada Dok 3, dan "temu" atau
"informasi" pada Dok 4. Term-term seperti "komputer" dan "data" yang muncul di beberapa dokumen sekaligus mendapatkan bobot IDF yang lebih
rendah sehingga skor TF-IDF-nya tidak setinggi term yang lebih spesifik. Hal ini penting karena term dengan bobot tinggi tersebut merupakan
kata kunci yang paling merepresentasikan topik unik dari masing-masing dokumen, sehingga sangat berguna untuk membedakan dan mengurutkan
(ranking) dokumen saat digunakan dalam perhitungan cosine similarity pada sistem pencarian.